# Záložný tréning v Google Colab / Kaggle (issue #28)

Hlavné tréningové prostredie je PC s AMD RX 9070 XT (WSL2, ROCm) – pozri `docs/gpu_prostredie.md`.
Tento notebook je **záloha**, keď PC nie je k dispozícii.

1. V Colabe: *Runtime → Change runtime type → GPU* (napr. T4).
2. Spúšťať bunky postupne zhora.

Projekt sa inštaluje **bez závislostí** (`--no-deps`): `uv.lock` zamyká pre Linux PyTorch s ROCm
(pre AMD), ale Colab má NVIDIA GPU – použije sa jeho predinštalovaný PyTorch s CUDA.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Repozitár a inštalácia

In [ ]:
%cd /content
!git clone -q https://github.com/SPSE-Zochova-2627-4c/Nosko_EKG-classification.git
%cd /content/Nosko_EKG-classification
!git log -1 --oneline

In [ ]:
# projekt bez závislostí + knižnice, ktoré Colab nemá predinštalované
# (--ignore-requires-python: projekt je overený na Pythone 3.11, Colab môže mať novší)
!pip install -q --no-deps --ignore-requires-python -e .
!pip install -q "wfdb>=4.1"

## 2. (Voliteľné) Ukladanie checkpointov na Google Drive

Colab po odpojení všetko zmaže. Ak sa má model zachovať, pripoj Drive a nastav cestu.

In [ ]:
import os

USE_DRIVE = False  # True = checkpointy a výstupy na Google Drive

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.environ["EKG_CHECKPOINT_DIR"] = "/content/drive/MyDrive/ekg/checkpoints"
    os.environ["EKG_OUTPUT_DIR"] = "/content/drive/MyDrive/ekg/outputs"

## 3. Overenie prostredia a rýchlosť GPU

In [ ]:
!ekg-check

In [ ]:
# CPU vs. GPU (float32; bfloat16 iba ak ho GPU podporuje – T4 nie)
!ekg-gpu-bench --steps 30

## 4. Dáta (PTB-XL, 1,84 GB)

Stiahnutie overí SHA256, preprocessing vytvorí cache (~1 GB). Výsledok je rovnaký ako na PC aj na Macu
(normalizácia sa počíta z train množiny podľa manifestu v gite).

In [ ]:
!ekg-download
!ekg-preprocess
!ekg-check-data

## 5. Tréning

Príklad: autoencoder (Model A). Po prerušení pokračovať s `--resume`.
Klasifikátor (Model B) pribudne v issue #16.

In [ ]:
!ekg-train-ae

In [ ]:
!ekg-ae-report

## 6. Zdieľanie natrénovaného modelu

Checkpointy nejdú do gitu – nahrajú sa ako GitHub Release `model-<run_name>` (súbory `best.pt` + prahy),
napr. z PC/Macu: `gh release create model-<run_name> checkpoints/<run_name>/best.pt ...`.
Z Colabu ich stačí stiahnuť (priečinok `checkpoints/`) alebo uložiť na Drive (krok 2).